In [1]:
import pandas as pd
import numpy as np

**CDC Overdose Data**
- Pick only the columns that I will want
- Is there a county FIPS field?

In [2]:
cdc = pd.read_csv("../data/raw/Underlying Cause of Death, 2018-2024, Single Race.csv")
cdc

,Notes,County,County Code,Deaths,Population,Crude Rate,Crude Rate Lower 95% Confidence Interval,Crude Rate Upper 95% Confidence Interval
0,NaN,"Autauga County, AL",1001.0,50.0,408275,12.2,9.1,16.1
1,NaN,"Baldwin County, AL",1003.0,427.0,1671387,25.5,23.1,28.0
2,NaN,"Barbour County, AL",1005.0,33.0,172769,19.1,13.1,26.8
3,NaN,"Bibb County, AL",1007.0,49.0,155538,31.5,23.3,41.6
4,NaN,"Blount County, AL",1009.0,106.0,412077,25.7,20.8,30.6
...,...,...,...,...,...,...,...,...
2549,certificates to include a cause of death code ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2550,"X60-X64, X85, and Y10-Y14). Jurisdictions can ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2551,"a result, users should consider that the actua...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2552,"deaths, with a crude rate of approximately 41....",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
cdc = cdc.drop(columns=[ 'Crude Rate', 
                        'Crude Rate Lower 95% Confidence Interval', 
                        'Crude Rate Upper 95% Confidence Interval',
                        'Notes'])
cdc.head(5)

,County,County Code,Deaths,Population
0,"Autauga County, AL",1001.0,50.0,408275
1,"Baldwin County, AL",1003.0,427.0,1671387
2,"Barbour County, AL",1005.0,33.0,172769
3,"Bibb County, AL",1007.0,49.0,155538
4,"Blount County, AL",1009.0,106.0,412077


In [4]:
cdc = cdc.rename(columns={
    "County": "county",
    "County Code": "fips",
    "Deaths": "deaths"
})

cdc["fips"] = cdc["fips"].astype(str).str.zfill(5)

In [5]:
cdc["deaths"] = cdc["deaths"].replace("Suppressed", "5")
cdc["deaths"] = pd.to_numeric(cdc["deaths"], errors="coerce")

In [6]:
cdc.to_csv(
    "../data/cleaned/cdc_cleaned_2024.csv",
    index=False
)

**Census Data**
- What variables did you pull?
- Do you have:
  - state
  - county
  - FIPS
  - income
  - poverty
  - unemployment

In [7]:
census = pd.read_csv('../data/raw/census_acs_2024.csv')
census.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3222 entries, 0 to 3221
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   county_name        3222 non-null   object 
 1   median_income      3222 non-null   int64  
 2   population         3222 non-null   int64  
 3   state              3222 non-null   int64  
 4   county             3222 non-null   int64  
 5   poverty_rate       3222 non-null   float64
 6   unemployment_rate  3222 non-null   float64
 7   fips               3222 non-null   int64  
dtypes: float64(2), int64(5), object(1)
memory usage: 201.5+ KB


In [8]:
census[['county', 'state']] = census['county_name'].str.split(',', expand=True)
census = census.drop(columns = ['county_name'])

census["fips"] = census["fips"].astype(str).str.zfill(5)

cols_to_back = ['median_income', 'population']

census = census[[col for col in census.columns if col not in cols_to_back] + cols_to_back]
census.head(5)

,state,county,poverty_rate,unemployment_rate,fips,median_income,population
0,Alabama,Autauga County,11.3,2.4,01001,72481,59947
1,Alabama,Baldwin County,10.1,3.0,01003,78775,246989
2,Alabama,Barbour County,21.4,7.8,01005,46042,24643
3,Alabama,Bibb County,22.5,12.1,01007,52541,22130
4,Alabama,Blount County,12.9,5.0,01009,64190,59518


In [9]:
census["median_income"] = pd.to_numeric(
    census["median_income"],
    errors="coerce"
)

census["population"] = pd.to_numeric(
    census["population"],
    errors="coerce"
)

In [10]:
census.isna().sum()

state                0
county               0
poverty_rate         0
unemployment_rate    0
fips                 0
median_income        0
population           0
dtype: int64

So here I only have median income and population for each county in the US. I want to get unemployment and poverty rates for each county. I will calculate that after merging.

In [11]:
census.to_csv(
    "../data/cleaned/census_clean_2024.csv",
    index=False
)

**SAMHSA Data**
Do I have:
 - facility name
 - state
 - county
 - ZIP
 - services offered

My main goal is to get a count of of the facilities per county that provide treatments for opioids.

In [12]:
facilities = pd.read_csv('../data/raw/2025_MH_Facilities_for_All_City_All.csv')
facilities.head(2)

,name1,name2,street1,street2,city,state,zip,phone,intake1,intake2,intake1a,intake2a,service_code_info
0,SpectraCare Health Systems,Henry County Clinic,219 Dothan Road,NaN,Abbeville,AL,36310.0,800-951-4357,NaN,NaN,NaN,NaN,SA MH SUMH * OP * CMHC * CHLOR FLUPH HALOP ARI...
1,SpectraCare Health Systems,Henry County Day Treatment,1242 U.S. Highway 431 South,NaN,Abbeville,AL,36310.0,800-951-4357,334-951-4357,NaN,NaN,NaN,MH SUMH * OP * CMHC * CHLOR FLUPH HALOP ARIPI ...


In [13]:
facilities['state'].eq('NJ').any()

np.True_

In [14]:
facilities["substance_abuse"] = facilities["service_code_info"].str.contains(
    r"\bSA\b",
    na=False
)

facilities["substance_abuse"].value_counts()

substance_abuse
True     10740
False     3538
Name: count, dtype: int64

In [15]:
facilities["methadone"] = facilities["service_code_info"].str.contains(
    "BUP",
    case=False,
    na=False
)

facilities["methadone"].value_counts()

methadone
False    14278
Name: count, dtype: int64

In [16]:
facilities["buprenorphine"] = facilities["service_code_info"].str.contains(
    "BUP",
    case=False,
    na=False
)

facilities["buprenorphine"].value_counts()

buprenorphine
False    14278
Name: count, dtype: int64

In [17]:
facilities["substance_abuse"] = facilities["service_code_info"].str.contains(
    r"\bSA\b",
    na=False
)

In [18]:
sa_facilities = facilities[facilities["substance_abuse"]]

In [19]:
sa_facilities['state'].eq('NJ').any()

np.True_

In [20]:
facilities["substance_abuse"] = facilities["service_code_info"].str.contains(
    r"\bSA\b",
    na=False
)

facilities["substance_abuse"].value_counts()

substance_abuse
True     10740
False     3538
Name: count, dtype: int64

Let's keep only the Substance abuse facilities.

In [21]:
facilities["substance_abuse"] = (
    facilities["service_code_info"]
    .str.contains(r"\bSA\b", na=False)
)

sa = facilities[
    facilities["substance_abuse"]
].copy()

print(sa.shape)

(10740, 16)


In [22]:
sa["zip"] = (
    sa["zip"]
    .astype(str)
    .str.extract(r"(\d{5})")[0]
)

sa = sa.dropna(subset=["zip"])

sa["zip"] = (
    sa["zip"]
    .astype(str)
    .str.zfill(5)
)

sa[["state","zip"]].head()

,state,zip
0,AL,36310
2,AL,35010
3,AL,36420
4,AL,36420
5,AL,36207


In [23]:
sa['state'].eq('NJ').any()

np.False_

Now we need to get a count of the facility per county based on Zip Code.

In [24]:
facility_saf_counts = (
    sa
    .groupby(
        ["state", "zip"]
    )
    .size()
    .reset_index(
        name="facility_count"
    )
)

facility_saf_counts['state'].eq('MA').any()

np.False_

In [25]:
facility_saf_counts.to_csv(
    "../data/cleaned/samhsa_facilities_zip.csv",
    index=False
)